# H4rmony base-versus-aligned threshold evaluation

Use a GPU runtime. Start with `caramel_sft`; it is the released SFT checkpoint and the cheapest comparison. The notebook scores exact **Yes** and **No** sequence probabilities rather than sampling answers. The reusable implementation lives in `scripts/harmony_eval/`.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/shengweiming/value-misalignment.git"
REPO_DIR = Path("/content/value-misalignment")

if not (REPO_DIR / ".git").exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", "main", REPO_URL, str(REPO_DIR)],
        check=True,
    )
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"],
    check=True,
)

In [ ]:
import torch
from scripts.harmony_eval import CHECKPOINT_PAIRS, run_checkpoint_pair

assert torch.cuda.is_available(), "Select Runtime > Change runtime type > GPU, then retry."
print("GPU:", torch.cuda.get_device_name(0))
for name, pair in CHECKPOINT_PAIRS.items():
    print(f"{name}: {pair.base_model}  ->  {pair.aligned_model}")

Choose one pair. `None` uses the catalog default: full precision for Caramel and matched 4-bit loading for each 7B pair.

In [ ]:
PAIR_NAME = "caramel_sft"  # caramel_sft, anthea_dpo, or breeze_dpo
COST_COUNTS = [0, 1, 10, 100, 1_000, 10_000, 100_000, 1_000_000]
BATCH_SIZE = 4
LOAD_IN_4BIT = None

In [ ]:
artifacts = run_checkpoint_pair(
    PAIR_NAME,
    cost_counts=COST_COUNTS,
    output_root=Path("/content/harmony_eval_outputs"),
    load_in_4bit=LOAD_IN_4BIT,
    batch_size=BATCH_SIZE,
)
print("Saved to:", artifacts.output_dir)

In [ ]:
import pandas as pd
from IPython.display import Image, display

display(pd.read_csv(artifacts.thresholds_path))
display(Image(filename=str(artifacts.plot_path)))

The single prompt in each family makes this an exploratory screen. A larger confirmatory run should add independently written, held-out variants before interpreting a threshold shift as radicalization.

In [ ]:
# Optional: download all artifacts from this run.
# import shutil
# from google.colab import files
# archive = shutil.make_archive(str(artifacts.output_dir), "zip", artifacts.output_dir)
# files.download(archive)